<a href="https://colab.research.google.com/github/noor00-ai/fly_rank_intern/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noor00-ai/fly_rank_intern/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector was created using ranking, engagement, traffic, and content-related signals available before the prediction point.

The selected features focus on decision-support analysis and exclude any fields that directly reveal future outcomes or label information.

Missing numeric values are handled using median imputation, while categorical values are handled using most frequent value imputation.

In [8]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


# Load dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)


# Selected features available before prediction
selected_features = [

    # Search demand signals
    "search_volume",
    "competition",
    "competition_level",
    "cpc",

    # Content characteristics
    "content_type",
    "main_intent",
    "word_count",
    "char_count",

    # Historical visibility signals
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",

    # Recent performance signals
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",

    # Content freshness
    "content_age_days",
    "days_since_last_update",

    # Engagement signals
    "ctr",
    "engagement_rate",
    "scroll_rate",

    # Trend signals
    "trend_pct",
    "trend_direction"
]


X = df[selected_features].copy()


print("Feature vector shape:", X.shape)

X.head()

Dataset shape: (30000, 44)
Feature vector shape: (30000, 24)


,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,impressions_90d,clicks_90d,...,impressions_last_30d,clicks_last_30d,sessions_last_30d,content_age_days,days_since_last_update,ctr,engagement_rate,scroll_rate,trend_pct,trend_direction
0,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,3803,29,...,578,2,2,187,20,0.76,5.88,4.55,-41.4,down
1,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,15320,7,...,2501,2,3,445,25,0.05,0.00,10.00,-57.7,down
2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,12581,11,...,2382,1,1,141,20,0.09,0.00,28.57,-60.9,down
3,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,11751,58,...,3626,22,35,463,22,0.49,1.28,3.45,-13.8,stable
4,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,19140,24,...,4211,10,14,263,14,0.13,0.00,24.29,-34.7,down


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The feature vector contains search, content, historical performance, engagement, freshness, and trend-related signals.

Search-related features:
- search_volume represents the estimated demand or search interest.
- competition and competition_level represent the difficulty of ranking.
- cpc represents commercial value indicators.

Content-related features:
- content_type and main_intent describe content characteristics.
- word_count and char_count represent content size and structure.

Historical performance features:
- impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, and engaged_sessions_90d represent previous visibility and user interaction.

Recent performance features:
- impressions_last_30d, clicks_last_30d, and sessions_last_30d capture recent changes in performance.

Freshness features:
- content_age_days and days_since_last_update represent content maturity and update recency.

Engagement features:
- ctr, engagement_rate, and scroll_rate measure user interaction signals.

Trend features:
- trend_pct and trend_direction represent recent performance movement.

Missing numeric values are handled using median imputation because it is robust against extreme values. Categorical features are encoded before modeling.

All selected features represent information available before the prediction decision. Fields that directly describe ranking outcomes or future information are excluded.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


missing_summary = (
    X.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_summary[missing_summary > 0]

,0
word_count,7699
char_count,7699
trend_pct,3388
competition_level,2610
cpc,2468
competition,2468
search_volume,2468
main_intent,2374
scroll_rate,125


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Leakage hunt:

A leakage audit was performed before modeling to identify features that could reveal the target outcome directly or contain information unavailable at prediction time.

Identifier fields such as content_id and client_id were excluded because they do not represent ranking signals.

Ranking outcome fields such as avg_position and position_tier were reviewed carefully because they are directly related to visibility outcomes. These fields were excluded from the feature vector to avoid unrealistic model performance.

Future-derived fields and post-event information were also excluded.

The remaining features represent observable search, content, engagement, freshness, and historical performance signals available before prediction.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check columns that may indicate leakage

possible_leakage_columns = [
    col for col in df.columns
    if any(keyword in col.lower()
           for keyword in [
               "position",
               "rank",
               "label",
               "target",
               "future",
               "outcome",
               "conversion"
           ])
]

print("Potential leakage-related columns:")
possible_leakage_columns

Potential leakage-related columns:


['avg_position', 'position_tier']

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded fields and reasons:

1. content_id
Reason: This is a unique identifier and does not represent a meaningful ranking signal.

2. client_id
Reason: This identifies client groups and is only used for grouped validation. It is excluded from model features to prevent memorization.

3. avg_position
Reason: This directly represents ranking performance and would leak the outcome information into the model.

4. position_tier
Reason: This is derived from ranking position and could create unrealistic model performance.

5. impression_tier
Reason: This is a derived visibility category and was excluded to avoid redundant information.

6. age_tier and freshness_tier
Reason: Raw freshness features provide more transparent signals, so derived categories were not used.

7. provider_used and model_used
Reason: These describe generation metadata and are not required for ranking signal analysis.

No client names, private identifiers, URLs, or sensitive information were used.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify excluded columns are not included in feature vector

excluded_columns = [
    "content_id",
    "client_id",
    "avg_position",
    "position_tier",
    "impression_tier"
]

for col in excluded_columns:
    print(f"{col}: Included in features -> {col in X.columns}")

content_id: Included in features -> False
client_id: Included in features -> False
avg_position: Included in features -> False
position_tier: Included in features -> False
impression_tier: Included in features -> False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.